# Overview
- WIKI dataset (face images + gender + age + ...)
  + Goal: classifying human gender
  + Input: a 64x64 image
  + Output: gender (female or male)
- Defining an CNN+MLP for WIKI dataset
- Training and evaluating an CNN+MLP

# Import libraries

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import time
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('> device:', device)

> device: cuda


In [3]:
# ===============================
#     PyTorch training helpers
# ===============================
class NumpyImageDataset(Dataset):
    """Wraps numpy arrays: images x of shape (N, H, W, C) and one-hot labels y of shape (N, num_classes)."""
    def __init__(self, x, y, transform=None):
        # channels-last (N, H, W, C) ---> channels-first (N, C, H, W)
        self.x = torch.tensor(np.ascontiguousarray(x.transpose(0, 3, 1, 2)), dtype=torch.float32)
        # one-hot vectors ---> class indices
        self.y = torch.tensor(np.argmax(y, axis=1), dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = self.x[idx]
        if self.transform is not None:
            img = self.transform(img)
        return img, self.y[idx]


def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over loader: trains when an optimizer is given, otherwise only evaluates. Returns (loss, acc)."""
    training = optimizer is not None
    model.train(training)
    total_loss, total_correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * xb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
    return total_loss / total, total_correct / total


def fit(model, train_loader, val_loader, criterion, optimizer, epochs, scheduler=None, callbacks=None):
    """Keras-like model.fit. Returns a history dict with keys 'loss', 'acc', 'val_loss', 'val_acc'.
    - scheduler: e.g. torch.optim.lr_scheduler.ReduceLROnPlateau, stepped with val_loss after every epoch
    - callbacks: callables cb(epoch, logs, model) called after every epoch; return True to stop training"""
    history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}
    callbacks = callbacks or []
    for epoch in range(epochs):
        t0 = time.time()
        loss, acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)
        logs = {'loss': loss, 'acc': acc, 'val_loss': val_loss, 'val_acc': val_acc}
        for k, v in logs.items():
            history[k].append(v)
        print('Epoch %d/%d - %.1fs - loss: %.4f - acc: %.4f - val_loss: %.4f - val_acc: %.4f - lr: %g'
              % (epoch + 1, epochs, time.time() - t0, loss, acc, val_loss, val_acc, optimizer.param_groups[0]['lr']))
        if scheduler is not None:
            scheduler.step(val_loss)
        stop = False
        for cb in callbacks:
            stop = bool(cb(epoch, logs, model)) or stop
        if stop:
            break
    return history


def evaluate(model, loader, criterion):
    """Like Keras model.evaluate: returns [loss, accuracy]."""
    return list(run_epoch(model, loader, criterion))


def predict(model, x):
    """Like Keras model.predict: x is a numpy array (N, H, W, C); returns softmax probabilities (N, num_classes)."""
    model.eval()
    with torch.no_grad():
        x = torch.tensor(np.ascontiguousarray(x.transpose(0, 3, 1, 2)), dtype=torch.float32, device=device)
        return F.softmax(model(x), dim=1).cpu().numpy()

In [3]:
num_classes = 2   # male vs. female

# (1) Data

### Download data at https://drive.google.com/drive/folders/0BxINLo5jshCRYW8xODhNSlkyLTQ?resourcekey=0-RNJvvSSE67Q1bMqQi3A1RQ&usp=sharing
This source code and npy files MUST be in the same location

In [4]:
x_train = np.load('64_64_11938_4098_train_x_onehot.npy')
y_train = np.load('64_64_11938_4098_train_y_onehot.npy')
x_train = np.rot90(x_train, k=3, axes=(1, 2))

x_test = np.load('64_64_5968_4098_val_x_onehot.npy')
y_test = np.load('64_64_5968_4098_val_y_onehot.npy')
x_test = np.rot90(x_test, k=3, axes=(1, 2))

# PyTorch datasets: images (N, 64, 64, 1) ---> (N, 1, 64, 64), one-hot labels ---> class indices
train_set = NumpyImageDataset(x_train, y_train)
test_set = NumpyImageDataset(x_test, y_test)

# (2) Declare model

In [6]:
def keras_init(m):
    # Keras default initialization: glorot_uniform weights, zero biases
    # (PyTorch's default init trains noticeably slower with the same SGD/Adam settings)
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)


class GenderCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # 64x64 portray image, input shape: (N, 1, 64, 64)

        # conv, pooling layers + dropout (padding='same' ---> padding=1 for 3x3 kernels)
        self.conv1_1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.conv_dropout1 = nn.Dropout(p=0.2)

        self.conv2_1 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.conv2_3 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.conv_dropout2 = nn.Dropout(p=0.2)

        self.conv3_1 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv3_3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2)
        self.conv_dropout3 = nn.Dropout(p=0.2)

        self.conv4_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.conv4_3 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool4 = nn.MaxPool2d(kernel_size=2)
        self.conv_dropout4 = nn.Dropout(p=0.2)

        self.conv5_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv5_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.pool5 = nn.MaxPool2d(kernel_size=2)
        self.conv_dropout5 = nn.Dropout(p=0.2)

        # similar to the MLP example!
        # matrix ---> vector: 64 / 2^5 = 2, so 256 x 2 x 2 = 1024 features
        self.flatten = nn.Flatten()

        # FC layers + dropout
        self.fc1 = nn.Linear(256 * 2 * 2, 1024)
        self.fc_dropout1 = nn.Dropout(p=0.2)

        self.fc2 = nn.Linear(1024, 1024)
        self.fc_dropout2 = nn.Dropout(p=0.2)

        # outputs raw logits: softmax is included in nn.CrossEntropyLoss (and applied in predict)
        self.fc3_10ways_softmax = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1_1(x))
        x = F.relu(self.conv1_2(x))
        x = self.conv_dropout1(self.pool1(x))

        x = F.relu(self.conv2_1(x))
        x = F.relu(self.conv2_2(x))
        x = F.relu(self.conv2_3(x))
        x = self.conv_dropout2(self.pool2(x))

        x = F.relu(self.conv3_1(x))
        x = F.relu(self.conv3_2(x))
        x = F.relu(self.conv3_3(x))
        x = self.conv_dropout3(self.pool3(x))

        x = F.relu(self.conv4_1(x))
        x = F.relu(self.conv4_2(x))
        x = F.relu(self.conv4_3(x))
        x = self.conv_dropout4(self.pool4(x))

        x = F.relu(self.conv5_1(x))
        x = F.relu(self.conv5_2(x))
        x = F.relu(self.conv5_3(x))
        x = self.conv_dropout5(self.pool5(x))

        x = self.flatten(x)

        x = self.fc_dropout1(F.relu(self.fc1(x)))
        x = self.fc_dropout2(F.relu(self.fc2(x)))
        return self.fc3_10ways_softmax(x)


# define model (Keras model name: 'mnist_mlp')
model = GenderCNN(num_classes).to(device)
model.apply(keras_init)

# print model summary
print(model)
print('Total params: {:,}'.format(sum(p.numel() for p in model.parameters())))
print('Trainable params: {:,}'.format(sum(p.numel() for p in model.parameters() if p.requires_grad)))

NameError: name 'num_classes' is not defined

In [8]:
# model architecture (replaces the Keras model_to_dot SVG graph)
print(model)

GenderCNN(
  (conv1_1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv1_2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_dropout1): Dropout(p=0.2, inplace=False)
  (conv2_1): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2_2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2_3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_dropout2): Dropout(p=0.2, inplace=False)
  (conv3_1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3_2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3_3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  

# (3) Train defined model
- Note that the training history including loss and accuracy will be saved in 'history' variable (a dict returned by our fit helper)
- In case your system runs out of memory (OOM), try to decrease batch size

In [9]:
# declare learning rate, loss function, and model metric
criterion = nn.CrossEntropyLoss()   # Keras 'categorical_crossentropy' (softmax is included)
lr = 0.0001
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# train the model
batch_size = 128
epochs = 30

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

starting_time = time.time()
history = fit(model, train_loader, test_loader, criterion, optimizer, epochs)
print('> training time is %.4f minutes' % ((time.time() - starting_time)/60))

C:\Users\MY PC\AppData\Roaming\Python\Python311\site-packages\torch\nn\modules\conv.py:560: UserWarning: nvmlInit_v2 failed; omitting per-process memory info from CUDA out-of-memory messages. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10\cuda\CUDACachingAllocator.cpp:1350.)
  return F.conv2d(


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 3.19 GiB is free. Of the allocated memory 17.51 MiB is allocated by PyTorch, and 4.49 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

### Training time
- (Times below were measured with the original Keras/TensorFlow version of this notebook)
- Training time with GTX 1080 is about 2.9352 minutes for 30 epochs
- Training time with Core i7 is around 109 minutes for 30 epochs

# (4) Evaluate trained model

In [ ]:
score = evaluate(model, test_loader, criterion)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
def int2gender(num):
    if num == 0:
        return 'female'
    else:
        return 'male'

In [ ]:
# select a test image randomly
random_test_index = np.random.choice(x_test.shape[0], size=1)[0]
test_img = x_test[random_test_index]
test_label = np.argmax(y_test[random_test_index])

# predict test image with trained model
pred_label = predict(model, np.expand_dims(test_img, axis=0))
pred_label = np.argmax(pred_label)

plt.imshow(test_img[:, :, 0], cmap='gray')
plt.title('true label = %s, predicted label = %s' % (int2gender(test_label), int2gender(pred_label)))
plt.show()